# 한 줄짜리 리서치 에이전트 만들기

리서치 업무는 전문가의 시간을 몇 시간씩 잡아먹습니다. 시장 분석가는 경쟁 정보를 손으로 모으고, 법무 팀은 규제 변화를 추적하고, 엔지니어는 여러 문서를 뒤지며 버그 리포트를 조사합니다. 핵심 난제는 정보를 찾는 것 자체가 아니라, 방금 알아낸 것을 바탕으로 다음에 무엇을 검색할지 아는 것입니다.

Claude Agent SDK를 사용하면 미리 정해진 워크플로 없이 외부 시스템을 자율적으로 탐색하는 에이전트를 만들 수 있습니다. 고정된 단계를 따르는 전통적인 워크플로 자동화와 달리, 리서치 에이전트는 찾아낸 것에 따라 전략을 바꿉니다. 유망한 단서를 따라가고, 상충하는 출처를 종합하며, 질문에 답하기에 충분한 정보를 모았을 때를 압니다.

## 이 쿡북을 마치면 다음을 할 수 있습니다.

- 몇 줄의 코드로 자율적으로 검색하고 정보를 종합하는 리서치 에이전트 만들기

이 토대는 필요한 정보가 미리 주어지지 않는 모든 작업에 적용됩니다. 경쟁 분석, 기술 문제 해결, 투자 리서치, 문헌 검토 같은 것들입니다.

# 왜 리서치 에이전트인가?

리서치가 에이전트에 이상적인 사용 사례인 이유는 두 가지입니다.

1. **정보가 그 안에 다 들어 있지 않습니다.** 입력 질문만으로는 답이 나오지 않습니다. 에이전트는 필요한 것을 모으기 위해 외부 시스템(검색 엔진, 데이터베이스, API)과 상호작용해야 합니다.
2. **경로는 탐색 도중에 드러납니다.** 워크플로를 미리 정할 수 없습니다. 에이전트가 기업 재무를 검색해야 할지 규제 공시를 검색해야 할지는 사업 모델에 대해 무엇을 알아내느냐에 달려 있습니다. 최적 전략은 조사 과정에서 스스로 모습을 드러냅니다.

가장 단순한 형태의 리서치 에이전트는 웹을 검색하고 결과를 종합합니다. 아래에서 Claude Agent SDK의 내장 웹 검색 도구로 그것을 단 몇 줄에 만들어 보겠습니다.

참고: [Claude Code의 내장 도구](https://docs.claude.com/en/docs/claude-code/settings#tools-available-to-claude) 전체 목록도 확인할 수 있습니다

# 사전 준비

이 가이드를 따라 하기 전에 다음을 확인하세요.

**필요한 사전 지식**

* Python 기초 — async/await, 함수, 기본 자료구조에 익숙할 것
* 에이전트 패턴에 대한 기본 이해 — 에이전트가 처음이라면 [효과적인 에이전트 만들기](https://www.anthropic.com/engineering/building-effective-agents)를 먼저 읽어 보시길 권합니다

**필요한 도구**

* Python 3.11 이상
* Anthropic API 키 [(여기서 발급)](https://console.anthropic.com)

**권장:**
* Claude Agent SDK 개념에 대한 친숙함
* LLM의 도구 사용 패턴에 대한 이해


## 준비

먼저 필요한 의존성을 설치합니다:

In [ ]:
%%capture
%pip install -U claude-agent-sdk python-dotenv

참고: .env 파일에 다음 내용이 들어 있는지 확인하세요.

```bash
ANTHROPIC_API_KEY=your_key_here
```

환경 변수를 불러오고 클라이언트를 설정합니다:

In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-opus-4-6"

## 첫 리서치 에이전트 만들기

가능한 가장 단순한 구현부터 시작하겠습니다. 웹을 검색하고 결과를 종합할 수 있는 리서치 에이전트입니다. Claude Agent SDK로는 단 몇 줄이면 됩니다.

핵심은 상태 없는 에이전트 상호작용을 만드는 query() 함수입니다. Claude에 WebSearch 도구 하나만 제공하고, 리서치 질문에 따라 언제 어떻게 그것을 쓸지 스스로 정하게 합니다.

In [3]:
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
)

from claude_agent_sdk import ClaudeAgentOptions, query

messages = []
async for msg in query(
    prompt="Research the latest trends in AI agents and give me a brief summary and relevant citiations links.",
    options=ClaudeAgentOptions(model=MODEL, allowed_tools=["WebSearch"]),
):
    print_activity(msg)
    messages.append(msg)

🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [4]:
display_agent_response(messages)

## 여기서 일어나는 일

- `query()`는 단일 턴 에이전트 상호작용을 만듭니다(대화 기억 없음)
- `allowed_tools=["WebSearch"]`는 승인을 묻지 않고 웹을 검색할 권한을 Claude에 줍니다
- 에이전트는 언제 검색할지, 어떤 질의를 던질지, 결과를 어떻게 종합할지 스스로 결정합니다

**`utils.agent_visualizer`의 시각화 유틸리티:**
- `print_activity()` — 에이전트의 동작(도구 호출, 사고)을 실시간으로 보여 줍니다
- `display_agent_response()` — 최종 응답을 꾸며진 HTML 카드로 렌더링합니다
- `visualize_conversation()` — 전체 대화의 타임라인 뷰를 만듭니다

이것으로 끝입니다! 단 몇 줄로 동작하는 리서치 에이전트가 만들어졌습니다. 에이전트는 관련 정보를 검색하고, 유망한 단서를 따라가며, 인용이 달린 종합 요약을 제공합니다.

query() 함수는 상태 없는 에이전트 상호작용을 만듭니다. 호출마다 독립적이어서 대화 기억도, 이전 질의의 맥락도 없습니다. 상태를 유지할 필요 없이 빠른 답이 필요한 일회성 리서치 작업에 안성맞춤입니다.

**도구 권한이 동작하는 방식:**

`allowed_tools=["WebSearch"]` 파라미터는 승인을 묻지 않고 검색할 권한을 Claude에 줍니다. 자율 동작에 결정적입니다.

- `허용된 도구` — Claude가 자유롭게 사용할 수 있습니다(여기서는 WebSearch)
- `그 밖의 도구` — 사용할 수는 있지만 사용 전에 승인이 필요합니다
- `읽기 전용 도구` — Read 같은 도구는 기본적으로 항상 허용됩니다
- `금지된 도구` — disallowed_tools에 도구를 추가하면 Claude의 컨텍스트에서 완전히 제거됩니다

**상태 없는 질의를 쓸 때:**

- 맥락이 중요하지 않은 일회성 리서치 질문
- 서로 무관한 리서치 작업의 병렬 처리
- 질의마다 새 컨텍스트를 원하는 경우

**상태 없는 질의를 쓰지 말아야 할 때:**

- 이전 발견 위에 쌓아 가는 멀티턴 조사
- 초기 결과를 바탕으로 한 반복적 개선
- 지속적인 맥락이 필요한 복잡한 분석

visualize_conversation 헬퍼로 에이전트가 실제로 무엇을 했는지 살펴보겠습니다:

In [5]:
from utils.agent_visualizer import visualize_conversation

visualize_conversation(messages)

## 프로토타입에서 프로덕션으로: 세 가지 핵심 개선

한 줄짜리 리서치 에이전트는 동작하지만 한계가 있습니다. 기억 없는 단발 질의로는 반복적 리서치("X를 찾고, 찾은 것을 바탕으로 Y를 분석해")를 처리할 수 없습니다. 구현을 더 개선할 세 가지 방법을 살펴보겠습니다.

**1. ClaudeSDKClient로 대화 기억 갖추기**: 상태 없는 질의는 이전 발견 위에 쌓을 수 없습니다. "최고의 AI 스타트업은?"이라고 물은 뒤 "그들은 어떻게 투자받았어?"라고 물으면, 두 번째 질의는 어떤 스타트업을 말하는지 전혀 모릅니다. `ClaudeSDKClient`를 사용하면 여러 질의에 걸쳐 대화 기록을 유지할 수 있습니다.


**2. 시스템 프롬프트로 전문화된 동작 만들기**: 리서치 영역마다 요구 사항이 다릅니다. 금융 분석에는 기술 뉴스 요약과는 다른 엄밀함이 필요합니다. 시스템 프롬프트에 리서치 기준, 선호 출처, 인용 형식, 출력 구조를 담으세요. 리서치에 특화된 예시는 [에이전트 프롬프팅 가이드](https://github.com/anthropics/anthropic-cookbook/tree/main/patterns/agents/prompts)를 참고하세요.

**3. Read 도구로 멀티모달 리서치 하기**: 실제 리서치는 텍스트만이 아닙니다. 시장 보고서에는 차트가, 기술 문서에는 도해가 있고, 경쟁 분석에는 스크린샷 비교가 필요합니다. `Read` 도구를 켜서 Claude가 이미지, PDF, 그 밖의 시각 자료를 분석하게 하세요.

이 세 가지 변경을 리서치 에이전트에 적용해 보겠습니다.

In [6]:
from claude_agent_sdk import ClaudeSDKClient

# System prompt with citation requirements for research quality
RESEARCH_SYSTEM_PROMPT = """You are a research agent specialized in AI.

When providing research findings:
- Always include source URLs as citations
- Format citations as markdown links: [Source Title](URL)
- Group sources in a "Sources:" section at the end of your response"""

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="research_agent",
        system_prompt=RESEARCH_SYSTEM_PROMPT,
        allowed_tools=["WebSearch", "Read"],
        max_buffer_size=10 * 1024 * 1024,  # Increase to 10MB for image handling
    )
) as research_agent:
    # First query: Analyze the chart image
    await research_agent.query("Analyze the chart in research_agent/projects_claude.png")
    async for msg in research_agent.receive_response():
        print_activity(msg)
        messages.append(msg)

    # Second query: Use web search to validate/contextualize the chart findings
    await research_agent.query(
        "Based on the chart analysis, search for recent news or data that validates or provides context for these findings. Include source URLs."
    )
    async for msg in research_agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: Glob()
✓ Tool completed
🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


### 🔧 큰 응답과 버퍼 한도 다루기

이미지나 큰 데이터를 다루다 보면 버퍼 초과 오류를 만날 수 있습니다:

```
Fatal error in message reader: Failed to decode JSON: JSON message exceeded maximum buffer size of 1048576 bytes
```

**왜 이런 일이 생길까요:**
- `max_buffer_size`의 기본값은 1MB(1,048,576바이트)입니다
- 이미지는 메시지 안에서 base64로 인코딩되어 크기가 크게 늘어납니다
- 디스크에서 약 200KB인 차트 이미지가 base64 인코딩 후 270KB 이상이 되고, 여기에 메시지 부가 정보가 더해집니다

**해결책:**
이미지나 큰 도구 출력을 다룰 때는 `ClaudeAgentOptions`의 `max_buffer_size`를 더 큰 값(예: 10MB)으로 설정하세요.

**모범 사례:**
- 용도에 맞게 버퍼 크기를 설정하세요. 일반적인 멀티모달 작업에는 10MB, 대용량 문서 처리에는 그 이상
- 정말 전체 이미지를 넘겨야 하는지 고민해 보세요. 설명이나 작은 썸네일로 충분할 때도 있습니다
- 버퍼 오류를 모니터링하고 그에 맞게 조정하세요
- 검증 가능한 리서치 결과를 위해 시스템 프롬프트에 인용 요구 사항을 포함하세요

## 여기서 일어나는 일

이 예제는 세 가지 개선을 모두 결합합니다. 대화 기억, 인용을 인식하는 시스템 프롬프트, 멀티모달 분석입니다.

**핵심 구성 요소:**

| 구성 요소 | 역할 |
|-----------|---------|
| `ClaudeSDKClient` | 여러 질의에 걸쳐 대화 상태를 유지합니다 |
| `RESEARCH_SYSTEM_PROMPT` | 인용 형식과 출처 URL을 강제합니다 |
| `allowed_tools=["WebSearch", "Read"]` | 웹 검색과 이미지·문서 분석을 가능하게 합니다 |
| `max_buffer_size=10MB` | base64 인코딩된 이미지를 넘침 없이 처리합니다 |

**실행 흐름:**

1. **첫 질의** — `Read` 도구로 차트 이미지를 분석합니다
2. **첫 응답 루프** — 에이전트가 끝낼 때까지 모든 메시지를 수집합니다
3. **두 번째 질의** — 웹을 검색해 차트에서 얻은 결과를 검증하고 맥락을 붙입니다
4. **맥락 상속** — 두 번째 질의가 첫 번째의 차트 분석을 기억합니다

**`query()` 대신 `ClaudeSDKClient`를 쓰는 이유:**

`async with ClaudeSDKClient()` 컨텍스트 관리자가 대화 상태를 유지합니다. `receive_response()` 호출마다 이전 맥락 위에 쌓입니다. 독립적이고 상태 없는 세션을 만드는 `query()`와 다른 점입니다.

In [7]:
visualize_conversation(messages)

Pattern,Implication
Claude Code dominates Startup Work,Developers building products at startups prefer the code-focused interface for rapid development
Claude.ai leads in educational contexts,"The conversational nature of Claude.ai makes it more approachable for learning, research, and coursework"
Personal Projects are universal,Both platforms serve individual developers working on side projects equally well
Enterprise usage is balanced,Both products have found their place in professional enterprise environments
Metric,2025 Data
AI-generated/assisted code,41% of all code globally
Developers using AI coding assistants,82% daily or weekly
Market size projection,$30.1 billion by 2032
Google's AI-assisted code,25%
Chart Finding,Validation Status


## 프로덕션을 위해 만들기

주피터 노트북은 배우기에 좋지만, 프로덕션 시스템에는 재사용 가능한 모듈이 필요합니다. 리서치 에이전트를 깔끔한 인터페이스로 `research_agent/agent.py`에 담아 두었습니다.

### 핵심 함수:

- `print_activity()` — 에이전트가 무엇을 하는지 실시간으로 보여 줍니다(공용 유틸리티에서 가져옴)
- `get_activity_text()` — 로깅이나 모니터링 같은 커스텀 핸들러를 위해 활동 텍스트를 추출합니다
- `send_query()` — 활동 표시가 내장된 리서치 질의의 주 진입점

### 내장된 모범 사례:

이 모듈에는 다음을 보장하는 `RESEARCH_SYSTEM_PROMPT`가 들어 있습니다.
- 출처 URL이 항상 인용으로 포함됩니다
- 인용이 깔끔하게 렌더링되도록 마크다운 링크로 형식화됩니다
- "Sources:" 섹션이 모든 참조를 모아 줍니다

### 표시 제어:

`send_query()` 함수에는 `display_result` 파라미터가 있습니다(기본값: `True`).
- `display_result=True` — 주피터 노트북에 꾸며진 HTML 카드를 렌더링합니다
- `display_result=False` — 프로그램에서 쓸 수 있도록 텍스트 결과만 반환합니다

이제 이 에이전트를 어떤 Python 스크립트에서도 쓸 수 있습니다!

대화 맥락이 중요하지 않은 독립적인 질문에 씁니다.

모듈이 다음을 자동으로 처리합니다.
- 실행 중 활동 표시
- 새 대화를 위한 컨텍스트 초기화
- 최종 응답의 꾸며진 HTML 렌더링

In [8]:
from research_agent.agent import send_query

# The module handles activity display, context reset, and result visualization internally
result = await send_query("What is the Claude Code SDK? Only do one websearch and be concise")

🤖 Using: WebSearch()
✓ Tool completed
🤖 Thinking...


이제 같은 대화를 재사용하는 멀티턴 대화를 시험해 봅니다.

멀티턴 대화도 매끄럽게 동작합니다. `continue_conversation=True`만 전달하면 됩니다:

In [9]:
result1 = await send_query("What is Anthropic? Only do one websearch and be concise")

🤖 Using: WebSearch()
✓ Tool completed
🤖 Thinking...


In [10]:
# Continue the conversation to dig deeper by setting continue_conversation=True
result2 = await send_query(
    "What are some of their products?",
    continue_conversation=True,
)

🤖 Thinking...


## 마무리

### 만든 것

이 쿡북에서 점점 정교해지는 리서치 에이전트 세 가지를 만들었습니다.

- 상태 없는 리서치 에이전트 — 독립적인 리서치 작업을 위한 한 줄짜리 질의
- 기억을 갖춘 상태 유지 에이전트 — 이전 발견 위에 쌓아 가는 멀티턴 조사
- 프로덕션 모듈 — 애플리케이션에 통합할 수 있는 재사용 가능한 리서치 함수

### 핵심 정리

**상태 없는 질의(query())를 쓸 때:**

- 독립적인 리서치 질문
- 서로 무관한 작업의 병렬 처리
- 매번 새 컨텍스트가 필요한 경우

**상태 유지 에이전트(ClaudeSDKClient)를 쓸 때:**

- 이전 발견 위에 쌓아 가는 멀티턴 조사
- 리서치의 반복적 개선
- 지속적인 맥락이 필요한 복잡한 분석

리서치 에이전트는 정보가 그 안에 다 들어 있지 않고 최적의 워크플로가 탐색 도중에 드러날 때 빛을 발합니다. 경쟁 분석, 기술 문제 해결, 문헌 검토, 탐사 보도가 모두 이 패턴에 해당합니다.

### 다음 단계

자율 리서치의 이 토대는 엔터프라이즈 수준 멀티에이전트 시스템으로 나아갈 준비를 해 줍니다. 다음 노트북에서는 다음을 배웁니다.

조율 에이전트 아래에서 전문 서브에이전트 오케스트레이션하기
훅과 커스텀 명령을 통한 거버넌스 구현하기
이해관계자(경영진 vs. 기술 팀)에 따라 출력 스타일 조정하기

다음: [01_The_chief_of_staff_agent.ipynb](01_The_chief_of_staff_agent.ipynb) — 단일 에이전트에서 멀티에이전트 오케스트레이션으로.